# BIOT 6900 — Module 1 Starter Notebook
### Computational Environment Setup & Tool Ecosystem

**Name:** _(Drashti Bhanushali)_  
**Date:** _(09/16/2026)_  

Work through this notebook top to bottom. Fill in every cell marked **TODO**, then commit the notebook to your `biot6900` GitHub repository and post the link on Canvas.

## How to use this notebook

- Run each cell in order with **Shift + Enter**.
- Cells marked **TODO** need something from you — an email, a dataset path, or a one-line note.
- If a cell errors, read the message, check the **Troubleshooting** section of the Lab Guide, or bring it to office hours.
- **Save often.** If you're on Colab, remember it forgets your files — commit to GitHub before you close the tab.
- **Platform:** Week 1 is designed to run **locally** in your `biot6900` conda environment. A Colab version is on Canvas if you need it.

## Part A — Environment verification
Confirm your environment is set up correctly before you go any further.

In [1]:
import sys, platform
print("Python version:   ", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Platform:         ", platform.platform())

Python version:    3.11.16
Python executable: /opt/anaconda3/envs/biot6900/bin/python3.11
Platform:          macOS-15.5-arm64-arm-64bit


In [2]:
import os
# You should see "biot6900". If not, activate it in a terminal: conda activate biot6900
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "not detected"))

Conda environment: biot6900


In [3]:
# Git version (shell command). If this errors, install Git and restart the notebook.
!git --version

git version 2.39.5 (Apple Git-154)


In [6]:
!conda --version

conda 26.5.3


In [5]:
import Bio, pandas as pd, numpy as np, requests
print("BioPython:", Bio.__version__)
print("pandas:   ", pd.__version__)
print("numpy:    ", np.__version__)
print("requests: ", requests.__version__)

BioPython: 1.88
pandas:    3.0.5
numpy:     2.4.6
requests:  2.34.2


**Checkpoint A:** all four package versions printed without error, and your environment shows `biot6900`. If anything failed, fix it before continuing.

## Part B — BioPython quickstart
A quick preview of the library you'll use all semester.

In [7]:
from Bio.Seq import Seq
dna = Seq("ATGGCCATTGTAATGGGCCGCTGA")
print("Sequence length:   ", len(dna))
print("Protein translation:", dna.translate())

Sequence length:    24
Protein translation: MAIVMGR*


**TODO (B):** In one sentence, what did `.translate()` do to the DNA sequence?

_Your answer:_ .translate() converted the DNA sequence into the amino acid sequence MAIVMGR, with at the end an asterisk is marked for the stop codon.

## Part C — Biomedical database API exploration
Run each query, then write your one-line note in the markdown cell that follows it. Each database uses the same rhythm: build a query, call the API, read the result.

### C1 — PubMed (via `Bio.Entrez`)

In [9]:
from Bio import Entrez
Entrez.email = "bhanushali.dr@northeastern.edu"   # TODO: NCBI requires your email
handle = Entrez.esearch(db="pubmed", term="AlphaFold protein structure", retmax=5)
record = Entrez.read(handle)
handle.close()
print("PubMed IDs:", record["IdList"])

PubMed IDs: ['42746578', '42742960', '42742337', '42741268', '42738854']


**TODO (C1) — Note:** which search term did you use, and how many PMIDs came back?

_Your note:_ I searched PubMed for “AlphaFold protein structure” and retrieved five PMIDs. We can use this search to help us identify literature that we can use to investigate a potential biological target.

### C2 — UniProt (REST API)

In [10]:
import requests
acc = "P04637"   # human tumor-suppressor p53; TODO: try another accession if you like
r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json")
data = r.json()
name = data["proteinDescription"]["recommendedName"]["fullName"]["value"]
print("Accession:", acc)
print("Protein:  ", name)

Accession: P04637
Protein:   Cellular tumor antigen p53


**TODO (C2) — Note:** which protein did you retrieve?

_Your note:_ I looked up the protein “Cellular tumor antigen p53” using its UniProt ID, P04637. UniProt can help me learn about a protein’s sequence and function when studying it as a possible drug target.

### C3 — PDB (RCSB REST API)

In [12]:
import requests
pdb_id = "1TUP"   # a p53 structure; TODO: try another PDB ID if you like
r = requests.get(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}")
print("PDB ID:  ", pdb_id)
print("Title:   ", r.json()["struct"]["title"])
print("Coordinates:", f"https://files.rcsb.org/download/{pdb_id}.pdb")

PDB ID:   1TUP
Title:    TUMOR SUPPRESSOR P53 COMPLEXED WITH DNA
Coordinates: https://files.rcsb.org/download/1TUP.pdb


**TODO (C3) — Note:** what structure did you retrieve?

_Your note:_ I looked up the structure 1TUP, which shows the tumor suppressor p53 bound to DNA. This structure can help me understand how p53 interacts with DNA.

### C4 — GWAS Catalog (EBI REST API)

In [13]:
import requests
base = "https://www.ebi.ac.uk/gwas/rest/api"
rsid = "rs7412"   # a well-known APOE variant; TODO: try another rsID if you like
r = requests.get(f"{base}/singleNucleotidePolymorphisms/{rsid}")
data = r.json()
print("Variant:        ", data.get("rsId"))
print("Functional class:", data.get("functionalClass"))

Variant:         rs7412
Functional class: missense_variant


**TODO (C4) — Note:** which variant did you look up, and one thing you noticed?

_Your note:_ I looked up the variant rs7412 and found that it is a missense variant, meaning it changes an amino acid in the protein.

In [14]:
print(data.get("_links", {}))

{'self': {'href': 'https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/rs7412'}, 'singleNucleotidePolymorphism': {'href': 'https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/rs7412{?projection}', 'templated': True}, 'associationsBySnpSummary': {'href': 'https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/rs7412/associations?projection=associationBySnp'}, 'currentSnp': {'href': 'https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/rs7412/currentSnp'}, 'associations': {'href': 'https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/rs7412/associations'}, 'studies': {'href': 'https://www.ebi.ac.uk/gwas/rest/api/singleNucleotidePolymorphisms/rs7412/studies'}}


In [17]:
association_url = data["_links"]["associationsBySnpSummary"]["href"]

response = requests.get(association_url, params={"size": 1}, timeout=30)
response.raise_for_status()

association_data = response.json()
print(association_data)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



## Part D — Download & load sample multi-omics data
Accessions and download links are posted on Canvas. Download your assigned file(s), then load each one and confirm its dimensions.

In [ ]:
import pandas as pd
# TODO: set the path to your downloaded file (use sep="\t" for tab-separated files)
path = "data/TODO_assigned_file.csv"
df = pd.read_csv(path)
print("Shape (rows, columns):", df.shape)
df.head()

**TODO (D) — Note:** for each dataset, record its shape and one sentence on what it contains.

_Your note:_ Part D not completed as instructed by dr sudhakaran

## Submission checklist

- [ ] Part A prints all versions and shows the `biot6900` environment
- [ ] Part C: all four queries ran, each with your one-line note
- [ ] Part D: dataset(s) loaded and shape printed
- [ ] Notebook committed to your `biot6900` GitHub repository
- [ ] Repository link posted on Canvas

_Graded 60 / 40: 60% clean execution, 40% interpretation & documentation._